# Sail-CV Tell-Tale Tracker (Colab)

This notebook runs the **tell-tale tracking pipeline** on a video with a predefined layout. It uses default parameters and lets you choose between:

- **Classifier** — bboxes + classifier labels (attached/detached/leech)
- **Vector** — bboxes + mask overlay + PCA direction arrows

**Before running:** Enable a GPU for faster inference: **Runtime → Change runtime type → T4 GPU** (or another GPU).

## 1. Clone repository and install dependencies

In [ ]:
import os
import sys

# Clone repo (use branch/main as needed)
REPO_URL = "https://github.com/estebanfoucher/Sail-CV.git"
PROJECT_DIR = "/content/Sail-CV"

if not os.path.isdir(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"Repo already cloned at {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install uv and sync dependencies (tracking + optional extras)
!pip install -q uv
!uv sync --all-extras

# Install ffmpeg (required for video I/O)
!apt-get update -qq && apt-get install -y -qq ffmpeg

print("Dependencies installed.")

## 2. Configuration

In [ ]:
# Choose pipeline mode: "classifier" (labels) or "vector" (masks + PCA arrows)
MODE = "classifier"  # or "vector"

PARAMETERS = {
    "classifier": "parameters/default_classifier.yml",
    "vector": "parameters/default_vector.yml",
}
parameters_file = PARAMETERS[MODE]
print(f"Using parameters: {parameters_file}")

## 3. Input: built-in sample or your own files

In [ ]:
from pathlib import Path

ROOT = Path(".")

# Option A: Use a built-in sample (C1 or C4). Both videos are in assets/tracking/DS_6/; C4 layout may need to be created first.
# Option B: Upload your own video and layout (set USE_BUILTIN = False and run the next cell)

USE_BUILTIN = True  # Set to False if you will upload files in the next cell
BUILTIN_SAMPLE = "C1"  # "C1" or "C4"

BUILTIN_VIDEOS = {
    "C1": (
        ROOT / "assets" / "tracking" / "DS_6" / "C1.mp4",
        ROOT / "assets" / "tracking" / "layouts" / "C1_layout.json",
    ),
    "C4": (
        ROOT / "assets" / "tracking" / "DS_6" / "C4.mp4",
        ROOT / "output" / "tracking_layouts" / "DS_6" / "C4_layout.json",
    ),
}

if USE_BUILTIN:
    video_path, layout_path = BUILTIN_VIDEOS[BUILTIN_SAMPLE]
    if not video_path.exists():
        raise FileNotFoundError(
            f"Built-in video not found: {video_path}. "
            "C1_fixture.mp4 and C4.mp4 may need to be present in the repo."
        )
    if not layout_path.exists():
        raise FileNotFoundError(
            f"Built-in layout not found: {layout_path}. "
            "For C4, create the layout first with:\n"
            "  uv run python src/tracking/annotate_layout_opencv.py --video assets/tracking/DS_6/C4.mp4 "
            "--time-sec 3 --output output/tracking_layouts/DS_6/C4_layout.json\n"
            "Or use the C1 sample (assets/tracking/DS_6/C1.mp4 + assets/tracking/layouts/C1_layout.json)."
        )
    print(f"Using built-in sample: {BUILTIN_SAMPLE}")
    print(f"  Video:  {video_path}")
    print(f"  Layout: {layout_path}")
else:
    # Paths will be set in the next cell after upload
    video_path = None
    layout_path = None
    print("Run the next cell to upload your video and layout JSON.")

In [ ]:
# Run this cell only if USE_BUILTIN is False: upload your video and layout file
if not USE_BUILTIN:
    from google.colab import files

    uploaded = files.upload()  # Pick your video and layout .json

    if len(uploaded) < 2:
        raise ValueError(
            "Please upload at least two files: one video (e.g. .mp4) and one layout (e.g. .json)"
        )
    # Separate by extension
    video_path = None
    layout_path = None
    for name, data in uploaded.items():
        path = Path(name)
        if path.suffix.lower() in (".mp4", ".avi", ".mov", ".mkv"):
            video_path = Path(name)
        elif path.suffix.lower() == ".json":
            layout_path = Path(name)
        with open(name, "wb") as f:
            f.write(data)
    if video_path is None or layout_path is None:
        raise ValueError(
            "Could not find one video file and one .json layout file among uploads."
        )
    print(f"Video:  {video_path}")
    print(f"Layout: {layout_path}")
else:
    print("Skipping upload (using built-in sample).")

## 4. Run the tracking pipeline

In [ ]:
import subprocess

out_dir = ROOT / "output" / "tracking"
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    "uv", "run", "python", "src/tracking/analyze_video.py",
    "--video", str(video_path.resolve()),
    "--layout", str(layout_path.resolve()),
    "--parameters", parameters_file,
    "--output", str(out_dir.resolve()),
]
result = subprocess.run(
    cmd, cwd=ROOT, capture_output=True, text=True
)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"Pipeline exited with code {result.returncode}")

## 5. View results

In [ ]:
from IPython.display import Video, display, FileLink

stem = video_path.stem
out_video = out_dir / f"{stem}_crop_module_tracked.mp4"
out_json = out_dir / f"{stem}_crop_module_tracked.json"

if out_video.exists():
    print("Output tracking video:")
    display(Video(str(out_video), width=640))
else:
    print(f"Video not found: {out_video}")

if out_json.exists():
    print("Download JSON results:")
    display(FileLink(str(out_json), result_html_prefix=""))
else:
    print(f"JSON not found: {out_json}")